# Normalization, the Hypersphere, and von Mises–Fisher Geometry — companion notebook

This notebook is the **narrative** half of the `hypersphere-vmf-geometry` Python pillar. The
reusable, tested implementation lives next door in [`hypersphere_vmf_geometry.py`](hypersphere_vmf_geometry.py);
here we import it and walk the topic section by section, so every claim the page makes renders as
executed output.

**One source of truth.** `hypersphere_vmf_geometry.py` owns the numbers — its `assert`-based harness
encodes the cosine–distance identity, the equatorial law, the von Mises–Fisher normalizing constant,
the mean resultant length, the maximum-entropy characterization, and the maximum-likelihood
estimators — and its `grid_table()` is mirrored *to the decimal* by `HypersphereLaboratory.tsx`. This
notebook never redefines that math; it calls into it.

Run from the repo root:

```
uv run --with numpy --with scipy --with jupyter \
    jupyter execute notebooks/hypersphere-vmf-geometry/01_hypersphere_vmf_geometry.ipynb
```


In [ ]:
import sys
import pathlib

# Make hypersphere_vmf_geometry.py importable whether the kernel starts in the repo
# root or in this notebook's own directory.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "hypersphere-vmf-geometry", pathlib.Path("notebooks/hypersphere-vmf-geometry")):
    if (_cand / "hypersphere_vmf_geometry.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
from hypersphere_vmf_geometry import (
    D_GRID, KAPPA_GRID, VMF_DISPLAY_DIM,
    normalize, sample_uniform_sphere, coordinate_marginal_samples, squared_distance,
    sample_vmf, log_vmf_norm_const, mean_resultant_length,
    mle_mu, kappa_hat_banerjee, kappa_hat_exact,
    coordinate_marginal_stats, equator_band_fraction, grid_table, finance_demo,
    test_cosine_distance_identity, test_coordinate_marginal, test_vmf_normalization,
    test_mean_resultant, test_max_entropy, test_mle_recovery, test_finance_clusters,
)

## 1. Why normalize: cosine similarity is the geometry of the sphere

Dense retrievers L2-normalize their embeddings, so every vector lives on the unit hypersphere
$S^{d-1}$. For unit vectors the squared Euclidean distance and the cosine similarity are tied by one
identity,

$$\lVert x - y\rVert^2 = \lVert x\rVert^2 - 2\langle x, y\rangle + \lVert y\rVert^2 = 2 - 2\langle x, y\rangle,$$

and $t \mapsto 2 - 2t$ is strictly decreasing. So ranking candidates by cosine similarity and ranking
them by (negated) Euclidean distance produce the **same order** — a retriever may train on one and
search with the other.

In [ ]:
d = 768
x = sample_uniform_sphere(5, d, seed=0)
y = sample_uniform_sphere(5, d, seed=1)
print("||x-y||^2 vs 2-2<x,y> (unit vectors, d=768):")
for a, b in zip(squared_distance(x, y), 2 - 2 * np.sum(x * y, axis=1)):
    print(f"  {a:.10f}  {b:.10f}")

# Ranking equivalence against a fixed query.
q = sample_uniform_sphere(1, 100, seed=7)[0]
cand = sample_uniform_sphere(6, 100, seed=8)
cos = cand @ q
print("\\nargsort by -cosine:", np.argsort(-cos))
print("argsort by  distance:", np.argsort(squared_distance(cand, np.broadcast_to(q, cand.shape))))

test_cosine_distance_identity()

## 2. The uniform distribution on the sphere and the equatorial band

What does "no information" look like on the sphere? The uniform distribution. Project a uniform random
unit vector $v$ onto any fixed axis $u$; the coordinate $t = \langle u, v\rangle$ has density

$$p(t) \;\propto\; (1 - t^2)^{(d-3)/2}, \qquad t \in [-1, 1],$$

with mean $0$ and variance **exactly $1/d$** — the very $1/d$ we met as $\operatorname{Var}\langle u, v\rangle$
in *high-dimensional geometry*, now read as a density. A clean fourth moment, $\mathbb E[t^4] = 3/(d(d+2))$,
pins down the shape. As $d$ grows the mass crowds the equator of every axis.

In [ ]:
print(f"{'d':>6}{'Var(t) emp':>13}{'1/d':>11}{'E[t^4] emp':>13}{'3/(d(d+2))':>13}{'band|t|<.1':>12}")
for d in (3, 10, 50, 200, 1536):
    s = coordinate_marginal_stats(d, n=60000, seed=2)
    print(f"{d:>6}{s['var']:>13.5f}{s['var_theory']:>11.5f}"
          f"{s['fourth']:>13.6f}{s['fourth_theory']:>13.6f}{s['band_frac']:>12.5f}")

test_coordinate_marginal()

## 3. The von Mises–Fisher distribution

A topical cluster on the sphere is not uniform — it concentrates around a mean direction $\mu$. The
**von Mises–Fisher** law is the natural model:

$$f(x; \mu, \kappa) = C_d(\kappa)\, e^{\kappa\, \mu^\top x}, \qquad
C_d(\kappa) = \frac{\kappa^{d/2 - 1}}{(2\pi)^{d/2} I_{d/2 - 1}(\kappa)},$$

where $I_\nu$ is the modified Bessel function of the first kind. The normalizer is a Bessel function
*precisely because* the surface integral $\int_{S^{d-1}} e^{\kappa \mu^\top x}\,dS$ reduces, via the
equatorial slice of §2, to the one-dimensional integral $\int_{-1}^{1} e^{\kappa t}(1 - t^2)^{(d-3)/2}\,dt$ —
which is the integral representation of $I_{d/2-1}(\kappa)$. We verify the reduction by checking that
$C_d(\kappa)$ integrates the density to $1$, and that our sampler (Wood 1994) produces unit vectors
whose mean direction is $\mu$.

In [ ]:
mu = normalize(np.arange(1.0, 11.0))   # a direction in R^10
x = sample_vmf(20000, mu, kappa=20.0, seed=3)
mu_hat, rbar = mle_mu(x)
print(f"vMF(mu, kappa=20) in R^10: all unit norm? {np.allclose(np.linalg.norm(x, axis=1), 1.0)}")
print(f"recovered mean direction cos(mu_hat, mu) = {mu_hat @ mu:.5f}")
print(f"log C_d(kappa) at (d=10, kappa=20) = {log_vmf_norm_const(10, 20.0):.5f}")

test_vmf_normalization()

## 4. Mean resultant length

The expected location of a vMF sample points along $\mu$ with magnitude

$$\rho = A_d(\kappa) = \frac{I_{d/2}(\kappa)}{I_{d/2-1}(\kappa)} = \lVert \mathbb E[x]\rVert,$$

the **mean resultant length**: $0$ when $\kappa = 0$ (uniform), rising to $1$ as $\kappa \to \infty$
(a point mass at $\mu$). We compute $A_d$ by the continued fraction of the Bessel ratio, so it stays
exact even at $d = 1536$ where $I_\nu(\kappa)$ itself underflows.

In [ ]:
print(f"{'kappa':>8}{'A_d (d=100)':>14}{'empirical ||mean||':>20}")
mu = normalize(np.ones(100))
for kappa in (5.0, 20.0, 100.0):
    emp = float(np.linalg.norm(sample_vmf(30000, mu, kappa, seed=11).mean(axis=0)))
    print(f"{kappa:>8.0f}{mean_resultant_length(100, kappa):>14.5f}{emp:>20.5f}")
print(f"\\nstable at the embedding dimension: A_1536(5000) = {mean_resultant_length(1536, 5000.0):.5f}")

test_mean_resultant()

## 5. von Mises–Fisher is the maximum-entropy law on the sphere

Why this density and not another? Among all distributions on $S^{d-1}$ with a prescribed mean
direction, vMF **maximizes entropy** — equivalently, it is the exponential family with sufficient
statistic $x$. On the circle we check this numerically: take the von Mises law $\propto e^{\kappa\cos\theta}$,
then build tilted competitors $\propto e^{a\cos\theta + b\cos 2\theta}$ tuned to the *same* mean
resultant, and confirm none has higher entropy.

In [ ]:
test_max_entropy()
print("von Mises maximizes entropy at fixed mean resultant (Gibbs check on the circle).")

## 6. Estimating the cluster: maximum likelihood for $\mu$ and $\kappa$

Given samples $x_1, \dots, x_n$ with resultant $R = \sum_i x_i$ and $\bar r = \lVert R\rVert/n$, the
maximum-likelihood estimates are

$$\hat\mu = \frac{R}{\lVert R\rVert}, \qquad A_d(\hat\kappa) = \bar r.$$

The mean-direction estimate is exact; the concentration solves an implicit Bessel-ratio equation. The
Banerjee et al. (2005) closed form $\hat\kappa \approx \bar r\,(d - \bar r^2)/(1 - \bar r^2)$ is an
**approximation** to that root — we report its error against the numerically-solved value rather than
treating it as exact.

In [ ]:
print(f"{'d':>6}{'kappa':>8}{'kappa_hat (exact)':>20}{'kappa_hat (Banerjee)':>22}")
for d in (10, 100, 768):
    for kappa in (10.0, 50.0, 200.0):
        rbar = mean_resultant_length(d, kappa)   # population resultant
        print(f"{d:>6}{kappa:>8.0f}{kappa_hat_exact(d, rbar):>20.4f}{kappa_hat_banerjee(d, rbar):>22.4f}")

test_mle_recovery()

## 7. The grid the viz mirrors

`grid_table()` prints the numbers `HypersphereLaboratory.tsx` mirrors to the decimal: the equatorial
panel's per-dimension coordinate variance and band fraction, and the vMF panel's per-$\kappa$ mean
resultant and $\hat\kappa$ round-trip at the display dimension.

In [ ]:
tbl = grid_table()
print("equatorial law:")
print(f"  {'d':>6}{'Var(t) emp':>13}{'1/d':>11}{'band|t|<0.1':>13}")
for r in tbl["equator"]:
    print(f"  {int(r['d']):>6}{r['proj_var']:>13.5f}{r['proj_var_theory']:>11.5f}{r['equator_band_frac']:>13.5f}")
print(f"\\nvMF at d={VMF_DISPLAY_DIM}:")
print(f"  {'kappa':>8}{'rho=A_d':>11}{'kappa_hat(Ban.)':>17}{'kappa_hat(exact)':>18}")
for r in tbl["vmf"]:
    print(f"  {r['kappa']:>8.0f}{r['rho']:>11.5f}{r['kappa_hat_banerjee']:>17.2f}{r['kappa_hat_exact']:>18.2f}")

## 8. Finance case study, and the summary

Two topical clusters of $1536$-dimensional financial-document embeddings — one tight
("interest-rate risk"), one loose ("supply-chain disruption") — modeled as von Mises–Fisher with
different $\kappa$. The tighter cluster has the higher mean resultant $\bar r$, the higher recovered
$\hat\kappa$, and a tighter cosine distribution, and the two clusters separate cleanly. These
embeddings are a **synthetic vMF stand-in**, deterministic and CPU-only, not a trained encoder.

Every claim above is also an `assert` in `hypersphere_vmf_geometry.py`; running this notebook end to
end re-verifies them all.

In [ ]:
test_finance_clusters()
print("\\nAll claims verified — the page, the laboratory, and this notebook agree by construction.")